# バレーボール動画 AI分析 - Step 1: Gemini API 動作確認

このノートブックは、Gemini API を使用して動画と選手写真からラリー分析を行う最小構成の実装です。

## Step 1 の範囲
- 1区間（3分）の動画を Gemini API に送信
- 選手写真を同時に送信
- JSON レスポンスを受け取り、パースする

## まだ実装しないもの
- 重複排除
- シート書き込み
- GAS 実装
- UI

## セル1: ライブラリインストール

In [ ]:
!pip install google-genai gdown -q

## セル2: インポート・設定

In [ ]:
import os
import json
from pathlib import Path

import google.genai as genai

try:
    from google.colab import auth, userdata
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

# ローカル実行用の秘密ファイル
LOCAL_SECRET_FILE = 'analysis/.local_secrets.json'


def load_local_secrets(file_path):
    """ローカルの秘密ファイルを読み込む"""
    candidates = [
        Path(file_path),
        Path('.local_secrets.json'),
        Path('analysis/.local_secrets.json'),
    ]

    for p in candidates:
        if p.exists():
            try:
                data = json.loads(p.read_text(encoding='utf-8'))
                if isinstance(data, dict):
                    print(f'ローカル秘密ファイルを読み込みました: {p}')
                    return data
            except Exception:
                pass

    return {}


def get_secret(key_name):
    """シークレット値を取得（Colab Secrets > 環境変数 > ローカルファイル）"""
    if IN_COLAB:
        try:
            value = userdata.get(key_name)
            if value is not None:
                return str(value).strip()
        except Exception:
            pass

    env_value = str(os.getenv(key_name, '')).strip()
    if env_value:
        return env_value

    file_value = LOCAL_SECRETS.get(key_name, '')
    return str(file_value).strip()


LOCAL_SECRETS = load_local_secrets(LOCAL_SECRET_FILE)

# Gemini API Key
GEMINI_API_KEY = get_secret('GEMINI_API_KEY')

if not GEMINI_API_KEY:
    raise ValueError(
        'GEMINI_API_KEY が設定されていません。\n'
        'Colab の場合は Secrets に GEMINI_API_KEY を登録してください。\n'
        'ローカルの場合は .local_secrets.json に {"GEMINI_API_KEY": "your-key"} を設定してください。'
    )

print('設定読み込み完了')

## セル3: Google Drive マウント

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive マウント完了')
else:
    print('ローカル実行モード: Drive マウントはスキップされます')

## セル4: Gemini クライアント初期化

In [ ]:
# Gemini クライアント初期化
client = genai.Client(api_key=GEMINI_API_KEY)

# モデル設定（設計書セクション4.4に準拠）
MODEL_NAME = "gemini-2.5-flash"
print(f'Gemini クライアント初期化完了: {MODEL_NAME}')

## セル5: Google Drive から動画をダウンロード

In [ ]:
import gdown

def download_video_from_drive(file_id, output_path):
    """Google Drive の共有リンクから動画をダウンロード（gdown使用）"""
    print(f'Google Drive から動画をダウンロード: {file_id}')
    
    url = f'https://drive.google.com/uc?id={file_id}'
    gdown.download(url, output_path, quiet=False)
    
    file_size = Path(output_path).stat().st_size
    print(f'ダウンロード完了: {output_path} ({file_size / 1024 / 1024:.2f} MB)')
    return output_path


# サンプル動画のファイルID
DRIVE_FILE_ID = "1iz8llxRGksr-jYpnwcy0FPvKeRAVjwQM"

# ダウンロード先パス
if IN_COLAB:
    DOWNLOAD_DIR = "/content"
else:
    DOWNLOAD_DIR = "analysis"

DOWNLOAD_PATH = f"{DOWNLOAD_DIR}/sample_video.mp4"

# 動画をダウンロード
download_video_from_drive(DRIVE_FILE_ID, DOWNLOAD_PATH)

## セル6: 動画・選手写真パス設定

In [ ]:
# ============================================
# テスト用ファイルパス設定
# ============================================

# 動画ファイル（Google Drive からダウンロードしたものを使用）
VIDEO_PATH = DOWNLOAD_PATH

# 選手写真ディレクトリ（写真がない場合は空リストにする）
# 写真を用意する場合は、適切なパスを設定してください
if IN_COLAB:
    PLAYERS_DIR = "/content/players"
else:
    PLAYERS_DIR = "analysis/players"

# 選手リスト（写真がない場合は空にする）
PLAYER_NAMES = []  # 写真を用意できたら ["ゆう", "りっきー", "れんれん"] などを設定

# 試合情報（ユーザープロンプト用）
MATCH_INFO = {
    "opponent": "レジン",
    "set": 1,
    "serve_team": "レジン",
    "rotation": 1,
    "segment_start_sec": 0,
    "segment_end_sec": 180
}

print(f'動画パス: {VIDEO_PATH}')
print(f'選手写真ディレクトリ: {PLAYERS_DIR}')
print(f'選手リスト: {PLAYER_NAMES}')

## セル7: 選手写真読み込み

In [ ]:
def load_player_photos(players_dir, player_names):
    """選手写真を読み込む"""
    photos = []
    for name in player_names:
        photo_path = Path(players_dir) / f"{name}.jpg"
        if not photo_path.exists():
            print(f'警告: 写真が見つかりません: {photo_path}')
            continue
        
        # 画像を Part オブジェクトとして読み込み
        with open(photo_path, 'rb') as f:
            photo_data = f.read()
        
        photos.append({
            'name': name,
            'mime_type': 'image/jpeg',
            'data': photo_data
        })
        print(f'写真読み込み: {name}')
    
    return photos


player_photos = load_player_photos(PLAYERS_DIR, PLAYER_NAMES)
print(f'\n選手写真読み込み完了: {len(player_photos)}枚')

## セル8: 動画を Gemini File API でアップロード

In [ ]:
def upload_video_to_gemini(video_path):
    """動画を Gemini File API でアップロード"""
    print(f'動画アップロード開始: {video_path}')
    
    video_file = client.files.upload(file=video_path)
    print(f'アップロード完了: {video_file.name}')
    print(f'URI: {video_file.uri}')
    
    # ファイルが ACTIVE 状態になるのを待つ
    print('ファイル処理完了を待機中...')
    import time
    while video_file.state == "PROCESSING":
        time.sleep(2)
        video_file = client.files.get(name=video_file.name)
        print(f'ファイル状態: {video_file.state}')
    
    if video_file.state == "FAILED":
        raise ValueError(f'ファイル処理に失敗しました: {video_file.state}')
    
    print(f'ファイル準備完了: {video_file.state}')
    
    return video_file


# 動画ファイルが存在するか確認
if not Path(VIDEO_PATH).exists():
    raise FileNotFoundError(f'動画ファイルが見つかりません: {VIDEO_PATH}')

video_file = upload_video_to_gemini(VIDEO_PATH)
print(f'\n動画 File URI: {video_file.uri}')

# システムプロンプト（設計書セクション6.3）
SYSTEM_PROMPT = """あなたはバレーボールの試合分析の専門家です。

## タスク
試合動画の一部（約3分間）を視聴し、全ラリーを検出・分析してください。

## チーム識別
- 画面手前側 = 「自チーム」、画面奥側 = 「相手」
- コートチェンジなし。ユニフォーム・背番号なし。コート位置のみで識別

## 選手識別
- ユーザープロンプトの選手写真を参照
- 服装ではなく体格・髪型・身長で識別。不明ならnull

## ラリーの定義
サーブのトスアップから得点確定まで。タイムアウト・休憩・選手交代は除外。

## 分析項目
- point_team（必須）: "自チーム" / "相手"
- deciding_team（必須）: ラリーを終わらせたプレーをしたチーム
- receive_grade: A/B/C/D/null（自チームサーブ側ならnull）
  - A: セッター定位置に正確
  - B: 1〜2歩移動
  - C: 大きく崩れた・二段トス
  - D: 直接失点
- receiver: 選手名/null
- play_type（必須）: サーブ/スパイク/フェイント/プッシュ/ブロック/ディグ/2段トス/フリーボール
- player: 選手名/null
- result_detail: アウト/ネット/シャット/タッチネット/ダブルコンタクト/サービスエース/タッチアウト/タッチイン/吸い込み/ポジション/オーバーネット/お見合い/その他/null
- attack_type: Aクイック/Bクイック/Cクイック/Aセミ/Bセミ/Cセミ/並行/オープン/2段トス/ブロードワイド/ブロードショート/null
- blocker_count: "0"/"1"/"2"/"3"/null
- zone_from/zone_to: 12ゾーン/null
- our_defense_type: "ブロック"/"ディグ"/null（相手攻撃で失点時のみ。不明なら"ディグ"）

## 確信度
confidence（全体）+ field_confidences（項目別12項目）。
0.8以上:明確 / 0.5〜0.79:おそらく / 0.5未満:推測（null推奨）

## タイムスタンプ
区間動画の先頭からの秒数。重複ラリーもそのまま出力。

## 出力形式
{
  "rallies": [{
    "rally_number": 整数,
    "rally_start_sec": 小数, "rally_end_sec": 小数,
    "confidence": 0.0-1.0,
    "point_team": "自チーム"/"相手",
    "deciding_team": "自チーム"/"相手",
    "receive_grade": .., "receiver": ..,
    "play_type": .., "player": ..,
    "result_detail": .., "attack_type": ..,
    "blocker_count": .., "zone_from": .., "zone_to": ..,
    "our_defense_type": .., "note": "..",
    "field_confidences": { 12項目 }
  }]
}

## 注意事項
- 不鮮明なら確信度を低く。推測よりnull
- タイムアウト・休憩はラリーとして検出しない
- 区間先頭/末尾でラリーが途中でもそのまま出力
"""

In [ ]:
# システムプロンプト（設計書セクション6.3）
SYSTEM_PROMPT = """あなたはバレーボールの試合分析の専門家です。

## タスク
試合動画の一部（約3分間）を視聴し、全ラリーを検出・分析してください。

## チーム識別
- 画面手前側 = 「自チーム」、画面奥側 = 「相手」
- コートチェンジなし。ユニフォーム・背番号なし。コート位置のみで識別

## 選手識別
- ユーザープロンプトの選手写真を参照
- 服装ではなく体格・髪型・身長で識別。不明ならnull

## ラリーの定義
サーブのトスアップから得点確定まで。タイムアウト・休憩・選手交代は除外。

## 分析項目
- point_team（必須）: "自チーム" / "相手"
- deciding_team（必須）: ラリーを終わらせたプレーをしたチーム
- receive_grade: A/B/C/D/null（自チームサーブ側ならnull）
- receiver: 選手名/null
- play_type（必須）: サーブ/スパイク/フェイント/プッシュ/ブロック/ディグ/2段トス/フリーボール
- player: 選手名/null
- result_detail: アウト/ネット/シャット/タッチネット/ダブルコンタクト/サービスエース/タッチアウト/タッチイン/吸い込み/ポジション/オーバーネット/お見合い/その他/null
- attack_type: Aクイック/Bクイック/Cクイック/Aセミ/Bセミ/Cセミ/並行/オープン/2段トス/ブロードワイド/ブロードショート/null
- blocker_count: "0"/"1"/"2"/"3"/null
- zone_from/zone_to: 12ゾーン/null
- our_defense_type: "ブロック"/"ディグ"/null（相手攻撃で失点時のみ。不明なら"ディグ"）

## 確信度
confidence（全体）+ field_confidences（項目別12項目）。
0.8以上:明確 / 0.5〜0.79:おそらく / 0.5未満:推測（null推奨）

## タイムスタンプ
区間動画の先頭からの秒数。重複ラリーもそのまま出力。

## 出力形式
{
  "rallies": [{
    "rally_number": 整数,
    "rally_start_sec": 小数, "rally_end_sec": 小数,
    "confidence": 0.0-1.0,
    "point_team": "自チーム"/"相手",
    "deciding_team": "自チーム"/"相手",
    "receive_grade": .., "receiver": ..,
    "play_type": .., "player": ..,
    "result_detail": .., "attack_type": ..,
    "blocker_count": .., "zone_from": .., "zone_to": ..,
    "our_defense_type": .., "note": "..",
    "field_confidences": { 12項目 }
  }]
}

## 注意事項
- 不鮮明なら確信度を低く。推測よりnull
- タイムアウト・休憩はラリーとして検出しない
- 区間先頭/末尾でラリーが途中でもそのまま出力
"""


# ユーザープロンプト（設計書セクション6.4）
def build_user_prompt(match_info):
    """ユーザープロンプトを構築"""
    prompt = f"""## 選手識別
"""
    
    # 選手写真の説明は画像Partとして別途渡すため、ここでは簡易的な説明のみ
    for i, name in enumerate(PLAYER_NAMES, 1):
        prompt += f"[写真{i}] この人は「{name}」です\n"
    
    prompt += """体格・髪型・身長で識別。不明ならnull。

## 試合情報
"""
    prompt += f"相手: {match_info['opponent']} / セット: {match_info['set']} / サーブ権: {match_info['serve_team']} / ローテーション: {match_info['rotation']}\n"
    
    prompt += """## 区間情報
"""
    prompt += f"セット全体の {match_info['segment_start_sec']}〜{match_info['segment_end_sec']} の区間。前区間と20秒オーバーラップ。\n"
    
    prompt += """## 指示
全ラリーを分析し、JSON形式で回答してください。"""
    
    return prompt


user_prompt = build_user_prompt(MATCH_INFO)
print('プロンプト構築完了')
print(f'\nユーザープロンプト:\n{user_prompt}')

## セル10: Gemini API 呼び出し（動画+写真+プロンプト）

In [ ]:
def call_gemini_with_video_and_photos(video_file, player_photos, system_prompt, user_prompt):
    """Gemini API を呼び出し（動画+選手写真+プロンプト）"""
    print('Gemini API 呼び出し開始...')
    
    # コンテンツを構築
    contents = [user_prompt]
    
    # 選手写真を追加
    for photo in player_photos:
        contents.append({
            'inline_data': {
                'mime_type': photo['mime_type'],
                'data': photo['data']
            }
        })
    
    # 動画を追加
    contents.append(video_file)
    
    # API 呼び出し
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=contents,
        config={
            "response_mime_type": "application/json",
            "max_output_tokens": 8192,
            "temperature": 0.2,
        }
    )
    
    print('API 呼び出し完了')
    return response


response = call_gemini_with_video_and_photos(video_file, player_photos, SYSTEM_PROMPT, user_prompt)
print(f'\n応答テキスト（先頭500文字）:\n{response.text[:500]}...')

## セル11: JSON パースと基本検証

In [ ]:
import json

def parse_and_validate_response(response_text):
    """JSON レスポンスをパースして基本検証"""
    try:
        data = json.loads(response_text)
    except json.JSONDecodeError as e:
        raise ValueError(f'JSON パースエラー: {e}') from e
    
    # 基本構造検証
    if 'rallies' not in data:
        raise ValueError('レスポンスに "rallies" キーがありません')
    
    if not isinstance(data['rallies'], list):
        raise ValueError('"rallies" は配列である必要があります')
    
    rallies = data['rallies']
    print(f'ラリー数: {len(rallies)}')
    
    # 各ラリーの必須フィールド検証
    required_fields = [
        'rally_number', 'rally_start_sec', 'rally_end_sec',
        'confidence', 'point_team', 'deciding_team', 'play_type'
    ]
    
    for i, rally in enumerate(rallies):
        for field in required_fields:
            if field not in rally:
                print(f'警告: ラリー{i}に必須フィールド "{field}" がありません')
    
    return data


parsed_data = parse_and_validate_response(response.text)
print('\nJSON パース・検証完了')
print(f'\nパースされたデータ（最初のラリー）:\n{json.dumps(parsed_data["rallies"][0] if parsed_data["rallies"] else {}, ensure_ascii=False, indent=2)}')

## セル12: アップロード動画削除

In [ ]:
def delete_video_file(video_file):
    """アップロードした動画ファイルを削除"""
    try:
        client.files.delete(name=video_file.name)
        print(f'動画ファイル削除完了: {video_file.name}')
    except Exception as e:
        print(f'警告: 動画ファイル削除に失敗: {e}')


delete_video_file(video_file)

## セル13: 結果表示

In [ ]:
print('=== Step 1 完了 ===')
print(f'検出されたラリー数: {len(parsed_data["rallies"])}')

for i, rally in enumerate(parsed_data['rallies'], 1):
    print(f"\n--- ラリー {i} ---")
    print(f"  タイムスタンプ: {rally['rally_start_sec']:.1f}s 〜 {rally['rally_end_sec']:.1f}s")
    print(f"  得点チーム: {rally['point_team']}")
    print(f"  決定チーム: {rally['deciding_team']}")
    print(f"  プレー種別: {rally['play_type']}")
    print(f"  確信度: {rally['confidence']:.2f}")